# 02 — Create the `cellflow` conda env
* env lives on /fast (`miniforge3/envs/cellflow`); pip cache and HF cache redirected to /fast/AG_Bunina/Pavel_1/data/cellflow
* `cellflow-tools==0.0.9` (PyPI) + `jax[cuda12]`; **no rapids-singlecell**
* `requests` added explicitly: cellflow 0.0.9 imports it at top level but only lists it in the `embedding` extra
* `TMPDIR` also redirected to /fast (node /tmp is 5 GB)
* env vars `XLA_PYTHON_CLIENT_PREALLOCATE=false`, `HF_HOME` set both as conda env vars (for `conda activate`) and in the Jupyter kernel spec

In [1]:
import os, subprocess
ROOT = "/fast/AG_Bunina/Pavel_1/miniforge3"
ENV = f"{ROOT}/envs/cellflow"
DATA = "/fast/AG_Bunina/Pavel_1/data/cellflow"
os.environ["PIP_CACHE_DIR"] = f"{DATA}/pip_cache"
os.environ["HF_HOME"] = f"{DATA}/hf_cache"
# node-local /tmp is only 5 GB — too small to unpack the CUDA wheels
os.environ["TMPDIR"] = f"{DATA}/tmp"
os.makedirs(os.environ["TMPDIR"], exist_ok=True)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

In [2]:
if not os.path.exists(f"{ENV}/bin/python"):
    !{ROOT}/bin/mamba create -y -q -p {ENV} python=3.11 pip 2>&1 | tail -3
!{ENV}/bin/python --version

Python 3.11.16


In [3]:
!{ENV}/bin/pip install -q "cellflow-tools==0.0.9" "jax[cuda12]" ipykernel nbconvert seaborn requests 2>&1 | tail -5

In [4]:
!{ROOT}/bin/conda env config vars set -p {ENV} XLA_PYTHON_CLIENT_PREALLOCATE=false HF_HOME={DATA}/hf_cache PIP_CACHE_DIR={DATA}/pip_cache
!{ROOT}/bin/conda env config vars list -p {ENV}

XLA_PYTHON_CLIENT_PREALLOCATE = false
HF_HOME = /fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache
PIP_CACHE_DIR = /fast/AG_Bunina/Pavel_1/data/cellflow/pip_cache


In [5]:
# register kernel, then inject env vars into kernel.json so they apply inside Jupyter too
import json, glob
!{ENV}/bin/python -m ipykernel install --user --name cellflow --display-name "Python (cellflow)"
kj = os.path.expanduser("~/.local/share/jupyter/kernels/cellflow/kernel.json")
spec = json.load(open(kj))
spec["env"] = {"XLA_PYTHON_CLIENT_PREALLOCATE": "false", "HF_HOME": f"{DATA}/hf_cache", "PIP_CACHE_DIR": f"{DATA}/pip_cache"}
json.dump(spec, open(kj, "w"), indent=1)
print(json.dumps(spec, indent=1))

Installed kernelspec cellflow in /home/p/pabrama/.local/share/jupyter/kernels/cellflow


{
 "argv": [
  "/fast/AG_Bunina/Pavel_1/miniforge3/envs/cellflow/bin/python",
  "-Xfrozen_modules=off",
  "-m",
  "ipykernel_launcher",
  "-f",
  "{connection_file}"
 ],
 "display_name": "Python (cellflow)",
 "language": "python",
 "metadata": {
  "debugger": true,
  "supported_encryption": "curve"
 },
 "kernel_protocol_version": "5.5",
 "env": {
  "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
  "HF_HOME": "/fast/AG_Bunina/Pavel_1/data/cellflow/hf_cache",
  "PIP_CACHE_DIR": "/fast/AG_Bunina/Pavel_1/data/cellflow/pip_cache"
 }
}


In [6]:
!{ENV}/bin/pip freeze > /fast/AG_Bunina/Pavel_1/cellflow-regen/requirements.lock.txt
!grep -iE "^(cellflow-tools|jax|jaxlib|jax-cuda12|flax|optax|ott-jax|anndata|scanpy|numpy|huggingface)" /fast/AG_Bunina/Pavel_1/cellflow-regen/requirements.lock.txt

anndata==0.12.19
cellflow-tools==0.0.9
flax==0.12.8
huggingface_hub==2.0.0
jax==0.10.2
jax-cuda12-pjrt==0.10.2
jax-cuda12-plugin==0.10.2
jaxlib==0.10.2
jaxopt==0.8.5
jaxtyping==0.3.11
numpy==2.4.6
optax==0.2.8
ott-jax==0.6.0
scanpy==1.11.5


In [7]:
# CPU-side smoke test (GPU verification is notebook 03, run on a GPU node)
!{ENV}/bin/python -c "import cellflow, jax; from importlib.metadata import version; print('cellflow-tools', version('cellflow-tools')); print('jax', jax.__version__); from cellflow.datasets import pbmc_cytokines; print('pbmc_cytokines OK')"
!du -sh {ENV}; du -sh ~/.cache 2>/dev/null

cellflow-tools 0.0.9
jax 0.10.2
pbmc_cytokines OK


6.8G	/fast/AG_Bunina/Pavel_1/miniforge3/envs/cellflow


695M	/home/pabrama/.cache
